In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!pip install -q xgboost

In [10]:
import os, json, re, pickle
import numpy as np
import pandas as pd

In [11]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

In [12]:
BIGO_DIR = '/content/drive/MyDrive/BIGO'
os.makedirs(BIGO_DIR, exist_ok=True)

In [13]:
import urllib.request

java_path = os.path.join(BIGO_DIR, 'java_data.jsonl')

rows = []
with open(java_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nClass distribution:')
print(df['complexity'].value_counts())
df.head(3)

Shape: (4900, 5)
Columns: ['src', 'complexity', 'problem', 'from', 'tags']

Class distribution:
complexity
linear       779
quadratic    765
constant     750
logn         700
nlogn        700
np           605
cubic        601
Name: count, dtype: int64


,src,complexity,problem,from,tags
0,import java.io.*;\n\n//@author Maurice Saldiva...,constant,0005_D,CODEFORCES,"implementation,math"
1,import java.util.*;\nimport static java.lang.M...,constant,0005_D,CODEFORCES,"implementation,math"
2,import java.util.*;\n\npublic class Rules {\n ...,constant,0005_D,CODEFORCES,"implementation,math"


In [14]:
FEATURE_NAMES = [
    "loc", "num_for", "num_while", "num_loops", "max_loop_depth",
    "nested2", "nested3", "num_sort", "num_hash_ds", "has_recursion",
    "num_log_loop", "num_binsearch", "num_shift", "num_methods",
]

In [15]:
def strip_code(code):
    code = re.sub(r"//.*", "", code)
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.S)
    code = re.sub(r'"(\\.|[^"\\])*"', '""', code)
    code = re.sub(r"'(\\.|[^'\\])*'", "''", code)
    return code

In [16]:
#depth

def _loop_depth(code):
    depth = 0
    loop_levels = []
    max_loop = 0
    nested2 = nested3 = 0
    pending = False
    for t in re.finditer(r"\bfor\b|\bwhile\b|\{|\}", code):
        s = t.group()
        if s in ("for", "while"):
            pending = True
        elif s == "{":
            depth += 1
            if pending:
                loop_levels.append(depth)
                cur = len(loop_levels)
                max_loop = max(max_loop, cur)
                if cur >= 2: nested2 += 1
                if cur >= 3: nested3 += 1
                pending = False
        elif s == "}":
            if loop_levels and loop_levels[-1] == depth:
                loop_levels.pop()
            depth = max(0, depth - 1)
    return max_loop, nested2, nested3

In [17]:
def hand_features(raw):
    code = strip_code(raw)
    loc = len([l for l in code.split("\n") if l.strip()])
    num_for = len(re.findall(r"\bfor\b", code))
    num_while = len(re.findall(r"\bwhile\b", code))
    num_loops = num_for + num_while
    num_sort = len(re.findall(r"\.sort\s*\(|Arrays\.sort|Collections\.sort", code))
    num_hash = len(re.findall(r"\b(HashMap|HashSet|TreeMap|TreeSet)\b", code))
    num_shift = len(re.findall(r"<<|>>", code))
    num_log = len(re.findall(r"[*/]=\s*2\b|<<=|>>=|\*\s*2\b|/\s*2\b", code))
    num_bin = len(re.findall(r"\bmid\b|binarySearch|\b(lo|hi|low|high)\b", code))
    num_methods = len(re.findall(r"\b(?:public|private|protected|static)[\w<>\[\]]*\s+\w+\s*\([^)]*\)\s*\{", code))
    names = re.findall(r"\b\w+\s+(\w+)\s*\([^)]*\)\s*\{", code)
    has_rec = 0
    for m in set(names):
        if len(re.findall(r"\b" + re.escape(m) + r"\s*\(", code)) > 1:
            has_rec = 1
            break
    mx, n2, n3 = _loop_depth(code)
    return [loc, num_for, num_while, num_loops, mx, n2, n3,
            num_sort, num_hash, has_rec, num_log, num_bin, num_shift, num_methods]


In [18]:
import pickle
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

In [19]:
X_hand = np.array([hand_features(c) for c in df["src"]], dtype=float)
corpus = [strip_code(c) for c in df["src"]]

In [20]:
le = LabelEncoder()
y = le.fit_transform(df["complexity"])
groups = df["problem"].values
idx = np.arange(len(df))

In [21]:
TOKEN = r"[A-Za-z_]\w+|[^\sA-Za-z0-9_]"

#done after Hyperpm tuning
PARAMS = dict(n_estimators=800, max_depth=6, learning_rate=0.1,
              subsample=0.8, colsample_bytree=0.8,
              tree_method="hist", eval_metric="mlogloss", n_jobs=-1)

In [22]:
def run_split(tr, te, name):
    tfidf = TfidfVectorizer(max_features=5000, token_pattern=TOKEN)
    Xtr = hstack([csr_matrix(X_hand[tr]), tfidf.fit_transform([corpus[i] for i in tr])]).tocsr()
    Xte = hstack([csr_matrix(X_hand[te]), tfidf.transform([corpus[i] for i in te])]).tocsr()
    clf = xgb.XGBClassifier(**PARAMS)
    clf.fit(Xtr, y[tr])
    acc = accuracy_score(y[te], clf.predict(Xte))
    print(f"\n=== {name} | accuracy = {acc:.4f} ===")
    print(classification_report(y[te], clf.predict(Xte), target_names=le.classes_))
    return acc

In [23]:
#Random split(Leakage)

tr, te = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
acc_random = run_split(tr, te, "RANDOM split")


=== RANDOM split | accuracy = 0.8388 ===
              precision    recall  f1-score   support

    constant       0.94      0.91      0.92       150
       cubic       0.90      0.94      0.92       120
      linear       0.69      0.74      0.71       156
        logn       0.82      0.86      0.84       140
       nlogn       0.90      0.74      0.81       140
          np       0.91      0.97      0.94       121
   quadratic       0.77      0.76      0.77       153

    accuracy                           0.84       980
   macro avg       0.85      0.85      0.84       980
weighted avg       0.84      0.84      0.84       980



In [24]:
#Problem split (honest)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(idx, y, groups))
acc_problem = run_split(tr, te, "PROBLEM split")
print(f"\nLEAKAGE GAP: random {acc_random:.3f} -> problem {acc_problem:.3f}")


=== PROBLEM split | accuracy = 0.5636 ===
              precision    recall  f1-score   support

    constant       0.84      0.89      0.87       214
       cubic       0.87      0.13      0.23       297
      linear       0.60      0.47      0.52       249
        logn       0.80      0.83      0.81       202
       nlogn       0.65      0.74      0.69       137
          np       0.09      1.00      0.17         6
   quadratic       0.07      0.63      0.13        27

    accuracy                           0.56      1132
   macro avg       0.56      0.67      0.49      1132
weighted avg       0.74      0.56      0.57      1132


LEAKAGE GAP: random 0.839 -> problem 0.564


In [25]:
tfidf_final = TfidfVectorizer(max_features=5000, token_pattern=TOKEN)
X_all = hstack([csr_matrix(X_hand), tfidf_final.fit_transform(corpus)]).tocsr()
final = xgb.XGBClassifier(**PARAMS); final.fit(X_all, y)
pickle.dump(final, open(os.path.join(BIGO_DIR,"xgb_model.pkl"), "wb"))
pickle.dump(tfidf_final, open(os.path.join(BIGO_DIR,"tfidf.pkl"), "wb"))
pickle.dump(le, open(os.path.join(BIGO_DIR,"label_encoder.pkl"), "wb"))
print("\nsaved pkls to", BIGO_DIR)


saved pkls to /content/drive/MyDrive/BIGO
